# NeMo Retriever Tutorial

This notebook demonstrates how to use NVIDIA NeMo Retriever for text reranking to improve RAG (Retrieval-Augmented Generation) pipeline quality.

Full documentation: [NeMo Retriever](https://docs.nvidia.com/nemo/retriever/latest/)

## Overview

This example demonstrates:
1. **Text Reranking**: Use NeMo Retriever to rerank search results
2. **RAG Integration**: Improve RAG pipeline by reranking retrieved documents
3. **API Usage**: How to call the retriever API endpoint
4. **Performance Comparison**: Compare retrieval results with and without reranking

**No API keys required!** The notebook uses your deployed NeMo Retriever NIM endpoint.

## Prerequisites

### Deployed Services
- ✅ **NeMo Retriever NIM**: `nv-rerankqa-1b-v2` service deployed
- ✅ **NeMo Entity Store** (optional, for RAG integration): `nemoentitystore-sample` service
- ✅ **Embedding NIM** (optional, for RAG integration): `nv-embedqa-1b-v2` service

### 🔒 Security Setup (REQUIRED FIRST STEP)

**IMPORTANT**: This notebook uses `env.donotcommit` file for sensitive configuration (tokens, API keys). 

**Before running this notebook:**
1. Copy the template: `cp env.donotcommit.example env.donotcommit`
2. Edit `env.donotcommit` and add your `NMS_NAMESPACE` (and other values as needed)
3. The `env.donotcommit` file is git-ignored and will NOT be committed to version control

**Find your namespace:**
```bash
oc projects
```

**Verify retriever service is deployed:**
```bash
oc get svc -n <your-namespace> | grep rerankqa
oc get pods -n <your-namespace> | grep rerankqa
```

In [ ]:
# ============================================================================
# CONFIGURATION: Load Environment Variables from env.donotcommit file
# ============================================================================
# 🔒 SECURITY: Never hardcode secrets in notebooks!
# All sensitive values (tokens, API keys) should be in env.donotcommit file
# 
# SETUP INSTRUCTIONS:
# 1. Copy env.donotcommit.example to env.donotcommit: cp env.donotcommit.example env.donotcommit
# 2. Edit env.donotcommit and fill in your values (especially NMS_NAMESPACE)
# 3. env.donotcommit is git-ignored and will NOT be committed to version control
#
# IMPORTANT: Run this cell FIRST before importing config!
# If you get connection errors, restart the kernel and run cells in order.
import os
import sys
from pathlib import Path

# Load env.donotcommit file from the notebook directory
try:
    from dotenv import load_dotenv
    # Find env.donotcommit file in the same directory as this notebook
    notebook_dir = Path().resolve()  # Current working directory (where notebook is run from)
    env_file = notebook_dir / "env.donotcommit"
    
    if env_file.exists():
        load_dotenv(env_file, override=False)  # override=False: don't overwrite existing env vars
        print(f"✅ Loaded env.donotcommit file from: {env_file}")
    else:
        print(f"⚠️  env.donotcommit file not found at: {env_file}")
        print(f"   📝 Please copy env.donotcommit.example to env.donotcommit and fill in your values:")
        print(f"      cp env.donotcommit.example env.donotcommit")
except ImportError:
    print("⚠️  python-dotenv not installed - install with: pip install python-dotenv")
    print("   Will use system environment variables only (not recommended)")

# Clear any cached config module to force reload
if 'config' in sys.modules:
    del sys.modules['config']
    print("⚠️  Cleared cached config module - will reload with new env vars")

# Set defaults (will be overridden by env.donotcommit file if present)
os.environ.setdefault("NMS_NAMESPACE", "anemo-rhoai")
os.environ.setdefault("RETRIEVER_TOP_K", "10")
os.environ.setdefault("RETRIEVER_TOP_N", "5")

print("\n✅ Environment variables loaded")
print(f"   NMS_NAMESPACE: {os.environ.get('NMS_NAMESPACE')}")
print(f"   Mode: Cluster (Workbench/Notebook)")
print(f"\n💡 If you see connection errors, restart the kernel and run cells in order!")

In [ ]:
# Install required packages
%pip install requests numpy python-dotenv

In [ ]:
# Load configuration
from config import (
    NIM_RETRIEVER_URL,
    ENTITY_STORE_URL,
    NIM_EMBEDDING_URL,
    NMS_NAMESPACE,
    RETRIEVER_TOP_K,
    RETRIEVER_TOP_N
)

print(f"✅ Configuration loaded")
print(f"Mode: Cluster (Workbench/Notebook)")
print(f"Retriever NIM: {NIM_RETRIEVER_URL}")
print(f"Namespace: {NMS_NAMESPACE}")
print(f"Top K: {RETRIEVER_TOP_K}, Top N: {RETRIEVER_TOP_N}")

## Part 1: Basic Text Reranking

NeMo Retriever reranks a list of candidate documents based on their relevance to a query. This improves retrieval quality by ordering results by relevance rather than just similarity.

In [ ]:
import requests
import json

# Example query and candidate documents
query = "What is machine learning?"

candidate_documents = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
    "The weather today is sunny with a temperature of 75 degrees Fahrenheit.",
    "Machine learning algorithms can be supervised, unsupervised, or reinforcement learning.",
    "Python is a popular programming language for data science and machine learning.",
    "Deep learning is a type of machine learning that uses neural networks with multiple layers.",
    "Cooking recipes often include ingredients like flour, sugar, and eggs.",
    "Natural language processing is an application of machine learning for text analysis.",
    "The capital of France is Paris, a beautiful city known for its art and culture."
]

print(f"Query: {query}")
print(f"\nCandidate Documents ({len(candidate_documents)}):")
for i, doc in enumerate(candidate_documents, 1):
    print(f"  {i}. {doc[:80]}...")

In [ ]:
# Call NeMo Retriever API for reranking
# The API endpoint is /v1/ranking (not /v1/rerank)
retriever_url = f"{NIM_RETRIEVER_URL}/v1/ranking"

# Convert documents to passages format (array of objects with "text" field)
passages = [{"text": doc} for doc in candidate_documents]

payload = {
    "model": "nvidia/llama-3.2-nv-rerankqa-1b-v2",
    "query": {"text": query},
    "passages": passages
}

headers = {
    "Content-Type": "application/json"
}

print(f"Calling retriever API: {retriever_url}")
print(f"Payload: {json.dumps(payload, indent=2)}")

try:
    response = requests.post(retriever_url, json=payload, headers=headers, timeout=30)
    response.raise_for_status()
    
    result = response.json()
    print(f"\n✅ Reranking successful!")
    print(f"Response: {json.dumps(result, indent=2)}")
    
    # Display reranked results
    # Response format: {"rankings": [{"index": 0, "logit": score}, ...], "usage": {...}}
    if "rankings" in result:
        # Sort by logit score (higher = more relevant) and take top N
        sorted_rankings = sorted(result["rankings"], key=lambda x: x["logit"], reverse=True)[:RETRIEVER_TOP_N]
        print(f"\n📊 Reranked Results (Top {RETRIEVER_TOP_N}):")
        for i, item in enumerate(sorted_rankings, 1):
            score = item.get("logit", "N/A")
            index = item.get("index", "N/A")
            print(f"\n  {i}. Score (logit): {score:.4f}, Index: {index}")
            print(f"     Document: {candidate_documents[index][:100]}...")
    
except requests.exceptions.RequestException as e:
    print(f"❌ Error calling retriever API: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"   Response status: {e.response.status_code}")
        print(f"   Response body: {e.response.text}")

## Part 2: RAG Pipeline Integration

Integrate NeMo Retriever with a RAG pipeline to improve retrieval quality. The workflow:
1. Use embedding model to retrieve initial candidate documents
2. Use retriever to rerank the candidates
3. Use top reranked documents as context for generation

In [ ]:
# Helper function to call embedding API
def get_embeddings(texts, embedding_url, input_type="passage"):
    """Get embeddings for a list of texts using the embedding NIM.
    
    Args:
        texts: List of text strings or single text string
        embedding_url: Base URL of the embedding service
        input_type: Either "passage" or "query" (default: "passage")
    
    Returns:
        List of embedding vectors, or None if error
    """
    url = f"{embedding_url}/v1/embeddings"
    
    # Ensure texts is a list
    if isinstance(texts, str):
        texts = [texts]
    
    # Process each text individually (API may not support batch)
    embeddings = []
    for text in texts:
        payload = {
            "input": text,
            "model": "nvidia/llama-3.2-nv-embedqa-1b-v2",
            "input_type": input_type
        }
        
        try:
            response = requests.post(url, json=payload, timeout=30)
            response.raise_for_status()
            result = response.json()
            # Extract embedding from response
            if "data" in result and len(result["data"]) > 0:
                embeddings.append(result["data"][0]["embedding"])
            else:
                print(f"⚠️  Unexpected response format: {result}")
                return None
        except Exception as e:
            print(f"Error getting embeddings: {e}")
            if hasattr(e, 'response') and e.response is not None:
                print(f"   Response: {e.response.text}")
            return None
    
    return embeddings

# Helper function to compute cosine similarity
import numpy as np

def cosine_similarity(vec1, vec2):
    """Compute cosine similarity between two vectors."""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print("✅ Helper functions defined")

In [ ]:
# Example: RAG pipeline with retriever reranking
query = "How does neural network training work?"

# Sample knowledge base documents
knowledge_base = [
    "Neural networks are trained using backpropagation algorithm that adjusts weights based on error gradients.",
    "Training data is split into training, validation, and test sets to evaluate model performance.",
    "Gradient descent is an optimization algorithm used to minimize the loss function during training.",
    "Overfitting occurs when a model learns training data too well and fails to generalize to new data.",
    "The learning rate controls how much weights are adjusted during each training iteration.",
    "Batch normalization helps stabilize training by normalizing inputs to each layer.",
    "Regularization techniques like dropout prevent overfitting by randomly disabling neurons during training.",
    "The weather forecast predicts rain for tomorrow afternoon.",
    "Activation functions like ReLU introduce non-linearity into neural network computations.",
    "Epochs refer to complete passes through the entire training dataset."
]

print(f"Query: {query}")
print(f"Knowledge Base: {len(knowledge_base)} documents")

In [ ]:
# Step 1: Initial retrieval using embeddings (if embedding service is available)
print("Step 1: Initial retrieval using embeddings...")

if NIM_EMBEDDING_URL:
    try:
        # Get embeddings for query and documents
        # Use input_type="query" for query embeddings, "passage" for document embeddings
        query_embedding = get_embeddings([query], NIM_EMBEDDING_URL, input_type="query")[0]
        doc_embeddings = get_embeddings(knowledge_base, NIM_EMBEDDING_URL, input_type="passage")
        
        # Compute similarities
        similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
        
        # Get top K candidates
        top_k_indices = np.argsort(similarities)[-RETRIEVER_TOP_K:][::-1]
        
        print(f"✅ Retrieved top {RETRIEVER_TOP_K} candidates using embeddings:")
        initial_candidates = [knowledge_base[i] for i in top_k_indices]
        for i, idx in enumerate(top_k_indices, 1):
            print(f"  {i}. Similarity: {similarities[idx]:.4f} - {knowledge_base[idx][:80]}...")
    except Exception as e:
        print(f"⚠️  Embedding retrieval failed: {e}")
        print("   Using all documents as candidates...")
        initial_candidates = knowledge_base
        top_k_indices = list(range(len(knowledge_base)))
else:
    print("⚠️  Embedding service not configured, using all documents as candidates...")
    initial_candidates = knowledge_base
    top_k_indices = list(range(len(knowledge_base)))

In [ ]:
# Step 2: Rerank candidates using NeMo Retriever
print(f"\nStep 2: Reranking top {len(initial_candidates)} candidates using NeMo Retriever...")

retriever_url = f"{NIM_RETRIEVER_URL}/v1/ranking"

# Convert candidates to passages format (array of objects with "text" field)
passages = [{"text": doc} for doc in initial_candidates]
payload = {
    "model": "nvidia/llama-3.2-nv-rerankqa-1b-v2",
    "query": {"text": query},
    "passages": passages,
}

try:
    response = requests.post(retriever_url, json=payload, timeout=30)
    response.raise_for_status()
    rerank_result = response.json()
    
    print(f"✅ Reranking successful!")
    
    # Display reranked results
    if "rankings" in rerank_result:
        print(f"\n📊 Final Reranked Results (Top {RETRIEVER_TOP_N}):")
        reranked_docs = []
        for i, item in enumerate(rerank_result["rankings"], 1):
            score = item.get("logit", item.get("logit", "N/A"))
            orig_idx = top_k_indices[item["index"]] if len(top_k_indices) > 0 else item["index"]
            doc = initial_candidates[item["index"]]
            reranked_docs.append(doc)
            print(f"\n  {i}. Relevance Score: {score:.4f}")
            print(f"     Original Index: {orig_idx}")
            print(f"     Document: {doc[:100]}...")
        
        print(f"\n✅ Top {RETRIEVER_TOP_N} documents selected for context generation")
        print(f"   These can now be used as context for LLM generation")
    else:
        print(f"⚠️  Unexpected response format: {rerank_result}")
        
except Exception as e:
    print(f"❌ Error during reranking: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"   Response: {e.response.text}")

## Part 3: Performance Comparison

Compare retrieval results with and without reranking to see the improvement.

In [ ]:
# Comparison example
test_query = "What are the key components of a neural network?"

test_docs = [
    "Neural networks consist of layers of interconnected neurons that process information.",
    "The input layer receives data, hidden layers perform computations, and output layer produces results.",
    "Weights and biases are parameters that neural networks learn during training.",
    "The restaurant serves Italian cuisine with fresh pasta and authentic sauces.",
    "Activation functions determine the output of each neuron based on its input.",
    "Backpropagation is used to update weights by propagating errors backward through the network.",
    "The weather is perfect for a picnic in the park today.",
    "Loss functions measure how far predictions are from actual values.",
    "Gradient descent optimizes neural network parameters by following the gradient of the loss function."
]

print(f"Query: {test_query}")
print(f"Total Documents: {len(test_docs)}\n")

# Without reranking (just embedding similarity)
if NIM_EMBEDDING_URL:
    try:
        # Use input_type="query" for query embeddings, "passage" for document embeddings
        query_emb = get_embeddings([test_query], NIM_EMBEDDING_URL, input_type="query")[0]
        doc_embs = get_embeddings(test_docs, NIM_EMBEDDING_URL, input_type="passage")
        similarities = [cosine_similarity(query_emb, doc_emb) for doc_emb in doc_embs]
        top_indices = np.argsort(similarities)[-RETRIEVER_TOP_N:][::-1]
        
        print("📊 Without Reranking (Embedding Similarity Only):")
        for i, idx in enumerate(top_indices, 1):
            print(f"  {i}. Similarity: {similarities[idx]:.4f} - {test_docs[idx][:70]}...")
    except Exception as e:
        print(f"⚠️  Embedding comparison failed: {e}")
        top_indices = list(range(min(RETRIEVER_TOP_N, len(test_docs))))
else:
    print("⚠️  Embedding service not available for comparison")
    top_indices = list(range(min(RETRIEVER_TOP_N, len(test_docs))))

In [ ]:
# With reranking
print(f"\n📊 With Reranking (NeMo Retriever):")

retriever_url = f"{NIM_RETRIEVER_URL}/v1/ranking"

# Convert documents to passages format (array of objects with "text" field)
passages = [{"text": doc} for doc in test_docs]

payload = {
    "model": "nvidia/llama-3.2-nv-rerankqa-1b-v2",
    "query": {"text": test_query},
    "passages": passages
}

try:
    response = requests.post(retriever_url, json=payload, timeout=30)
    response.raise_for_status()
    result = response.json()
    
    # Response format: {"rankings": [{"index": 0, "logit": score}, ...], "usage": {...}}
    if "rankings" in result:
        # Sort by logit score (higher = more relevant) and take top N
        sorted_rankings = sorted(result["rankings"], key=lambda x: x["logit"], reverse=True)[:RETRIEVER_TOP_N]
        print(f"✅ Reranking Results:")
        for i, item in enumerate(sorted_rankings, 1):
            score = item.get("logit", "N/A")
            idx = item.get("index", "N/A")
            print(f"  {i}. Relevance (logit): {score:.4f} - {test_docs[idx][:70]}...")
        
        print(f"\n💡 Notice how reranking improves relevance ordering!")
    else:
        print(f"⚠️  Unexpected response format: {result}")
        
except Exception as e:
    print(f"❌ Reranking failed: {e}")


## Summary

This tutorial demonstrated:

1. **Basic Reranking**: How to use NeMo Retriever to rerank candidate documents
2. **RAG Integration**: How to combine embedding-based retrieval with retriever reranking
3. **Performance Comparison**: The improvement in relevance ordering with reranking

### Key Benefits of NeMo Retriever:

- **Improved Relevance**: Reranking provides better relevance scores than embedding similarity alone
- **Better Context**: Top reranked documents provide more relevant context for LLM generation
- **Flexible Integration**: Works with any embedding-based retrieval system
- **Easy to Use**: Simple REST API for reranking operations

### Next Steps:

- Integrate retriever into your RAG pipeline
- Experiment with different `top_n` values
- Combine with Entity Store for production RAG systems
- Use reranked results as context for chat completions

For more information, see the [NeMo Retriever documentation](https://docs.nvidia.com/nemo/retriever/latest/).